<a href="https://colab.research.google.com/github/HemanthM17/Gen-Ai-Assignment/blob/main/TASK_3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Chatbot using Hugging Face Transformers

## Step 1: Install Required Libraries

In [ ]:
!pip install transformers torch -q

## Step 2: Import Libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('✅ Libraries imported successfully.')
print(f'PyTorch version      : {torch.__version__}')

# Use GPU if available, otherwise CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device being used    : {device.upper()}')

## Step 3: Load Pre-trained Transformer Model

In [ ]:
MODEL_NAME = 'microsoft/DialoGPT-medium'

print(f'Loading model: {MODEL_NAME}')
print('This may take a minute on first run (downloading model weights)...\n')

# Load tokenizer — converts text to tokens the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')

# Load the pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()  # Set to evaluation mode — disables dropout

print('✅ Model and tokenizer loaded successfully.')
print(f'Model parameters     : ~345 million')
print(f'Model type           : GPT-2 based Causal LM')
print(f'Running on           : {device.upper()}')

## Step 4: Build Response Generation Function

In [ ]:
def generate_response(user_input, chat_history_ids=None, max_length=1000):
    """
    Generate a chatbot response using DialoGPT.

    Args:
        user_input       : The message typed by the user
        chat_history_ids : Tensor of previous conversation tokens (None for first turn)
        max_length       : Maximum total token length for conversation history

    Returns:
        response         : The chatbot's reply as a string
        new_history_ids  : Updated conversation history tensor
    """

    # Encode user input and append end-of-string token
    # EOS token acts as a separator between turns in DialoGPT
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    # Append new input to conversation history
    # On first turn, history is just the new input
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Trim conversation if it gets too long (model has a max context window)
    if bot_input_ids.shape[-1] > max_length:
        bot_input_ids = bot_input_ids[:, -max_length:]

    # Generate response using the model
    # do_sample=True  : allows creative, varied responses
    # temperature     : controls randomness (lower = more focused)
    # top_p           : nucleus sampling — only consider top 92% probability tokens
    # top_k           : limit vocabulary to top 50 tokens at each step
    with torch.no_grad():
        new_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens    = 150,
            do_sample         = True,
            temperature       = 0.75,
            top_p             = 0.92,
            top_k             = 50,
            pad_token_id      = tokenizer.eos_token_id,
            repetition_penalty = 1.3
        )

    # Decode only the newly generated tokens (not the entire history)
    response = tokenizer.decode(
        new_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    # Fallback if model generates an empty response
    if not response:
        response = "I'm not sure I understand. Could you rephrase that?"

    return response, new_history_ids


print('✅ generate_response() function defined.')

## Step 5: Build the Interactive Chatbot

In [ ]:
def run_chatbot():
    """
    Interactive console-based chatbot.
    Maintains conversation history across multiple turns.
    Type 'exit' or 'quit' to end the session.
    """

    print('=' * 60)
    print('       🤖  AI CHATBOT  —  Powered by DialoGPT')
    print('=' * 60)
    print('Chatbot: Hello! I am your AI assistant.')
    print('         How can I help you today?')
    print('         (Type "exit" or "quit" to end the conversation)')
    print('-' * 60)

    chat_history_ids = None   # Stores the full conversation context
    turn_count       = 0      # Track number of conversation turns

    while True:
        # Get user input
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nChatbot: Session ended. Goodbye! 👋')
            break

        # Skip empty inputs gracefully
        if not user_input:
            print('Chatbot: Please type something so I can help you.')
            continue

        # Check for exit conditions
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print('Chatbot: Thank you for chatting with me!')
            print('         It was great talking to you. Goodbye! 👋')
            print('=' * 60)
            break

        # Generate and display response
        response, chat_history_ids = generate_response(
            user_input, chat_history_ids
        )

        turn_count += 1
        print(f'Chatbot: {response}')
        print('-' * 60)


print('✅ run_chatbot() function defined.')
print('\nRun the next cell to start chatting!')

## Step 6: Start the Chatbot

In [ ]:
run_chatbot()

       🤖  AI CHATBOT  —  Powered by DialoGPT
Chatbot: Hello! I am your AI assistant.
         How can I help you today?
         (Type "exit" or "quit" to end the conversation)
------------------------------------------------------------
You: how are you
Chatbot: Good! How about yourself?
------------------------------------------------------------

Chatbot: Session ended. Goodbye! 👋


## Step 7: Demo — Simulated Conversation Output

In [ ]:
def simulate_conversation(messages):
    """
    Simulate a full multi-turn conversation for demonstration.
    Shows the chatbot maintaining context across multiple turns.
    """
    print('=' * 60)
    print('  🤖  SIMULATED CONVERSATION DEMO')
    print('=' * 60)
    print('Chatbot: Hello! I am your AI assistant. How can I help you today?')
    print('-' * 60)

    chat_history_ids = None

    for message in messages:
        print(f'You    : {message}')
        response, chat_history_ids = generate_response(message, chat_history_ids)
        print(f'Chatbot: {response}')
        print('-' * 60)

    print('You    : exit')
    print('Chatbot: Thank you for chatting with me! Goodbye! 👋')
    print('=' * 60)


demo_messages = [
    "Hello!",
    "What is Artificial Intelligence?",
    "Can you give me an example of machine learning?",
    "What is the difference between AI and ML?",
    "That is very helpful, thank you!"
]

simulate_conversation(demo_messages)

  🤖  SIMULATED CONVERSATION DEMO
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------
You    : Hello!
Chatbot: I'm sorry, I couldn't find that in the album.
------------------------------------------------------------
You    : What is Artificial Intelligence?
Chatbot: Yes! Thanks for asking about it.
------------------------------------------------------------
You    : Can you give me an example of machine learning?
Chatbot: Machine Learning Machine learning
------------------------------------------------------------
You    : What is the difference between AI and ML?
Chatbot: ML stands for Linear Algorithms. AI stands for artificial intelligence.
------------------------------------------------------------
You    : That is very helpful, thank you!
Chatbot: Happy to help!
------------------------------------------------------------
You    : exit
Chatbot: Thank you for chatting with me! Goodbye! 👋


## Step 8: Understanding How the Model Works

In [ ]:
print('📋 MODEL INFORMATION')
print('=' * 50)
print(f'Model Name     : microsoft/DialoGPT-medium')
print(f'Architecture   : GPT-2 (Transformer Decoder)')
print(f'Parameters     : ~345 million')
print(f'Context Window : 1024 tokens')
print(f'Training Data  : 147M Reddit conversations')
print(f'Task           : Causal Language Modelling')
print(f'Device         : {device.upper()}')

print('\n📋 TOKENIZER INFORMATION')
print('=' * 50)
print(f'Vocabulary Size: {tokenizer.vocab_size}')
print(f'EOS Token      : {tokenizer.eos_token!r} (ID: {tokenizer.eos_token_id})')
print(f'Padding Side   : {tokenizer.padding_side}')

sample = "What is Machine Learning?"
tokens = tokenizer.encode(sample)
print(f'\nSample text    : "{sample}"')
print(f'Token IDs      : {tokens}')
print(f'Token count    : {len(tokens)}')
print(f'Decoded back   : "{tokenizer.decode(tokens)!r}"')

📋 MODEL INFORMATION
Model Name     : microsoft/DialoGPT-medium
Architecture   : GPT-2 (Transformer Decoder)
Parameters     : ~345 million
Context Window : 1024 tokens
Training Data  : 147M Reddit conversations
Task           : Causal Language Modelling
Device         : CPU

📋 TOKENIZER INFORMATION
Vocabulary Size: 50257
EOS Token      : '<|endoftext|>' (ID: 50256)
Padding Side   : left

Sample text    : "What is Machine Learning?"
Token IDs      : [2061, 318, 10850, 18252, 30]
Token count    : 5
Decoded back   : "'What is Machine Learning?'"


In [ ]:
import matplotlib.pyplot as plt

test_turns = [
    "Hello!",
    "What is deep learning?",
    "How does it relate to neural networks?",
    "Can you give me a simple example?"
]

history      = None
token_counts = []

for turn in test_turns:
    _, history = generate_response(turn, history)
    token_counts.append(history.shape[-1])

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(test_turns)+1), token_counts, marker='o',
         color='#0D9488', linewidth=2, markersize=8)
plt.fill_between(range(1, len(test_turns)+1), token_counts, alpha=0.15, color='#0D9488')
plt.title('Conversation History Growth (tokens per turn)', fontweight='bold', fontsize=12)
plt.xlabel('Turn Number')
plt.ylabel('Total Tokens in Context')
plt.xticks(range(1, len(test_turns)+1), [f'Turn {i}' for i in range(1, len(test_turns)+1)])
plt.tight_layout()
plt.show()

print('As more turns happen, the context window fills up.')
print('The model uses ALL previous turns to generate each new response.')

## Step 9: How Transformers Work — Conceptual Understanding

### What is DialoGPT?

DialoGPT is based on the **GPT-2** architecture — a **Transformer Decoder** model trained using causal language modelling. It predicts the next token given all previous tokens.

**Key concepts:**

| Concept | What it means |
|---|---|
| Tokenization | Text is split into subword units called tokens |
| Embeddings | Each token is mapped to a dense numerical vector |
| Self-Attention | The model weighs how much each token should influence others |
| Causal Masking | The model can only look at past tokens, not future ones |
| Text Generation | The model predicts one token at a time, sampling from probabilities |

### Why DialoGPT instead of plain GPT-2?

GPT-2 was trained to complete text (like a story continuation). DialoGPT was **fine-tuned specifically on conversations** — so it has learned to respond in a conversational style rather than just continuing text.

### Generation Parameters Explained

| Parameter | Value Used | Effect |
|---|---|---|
| `temperature` | 0.75 | Controls randomness — lower = more focused, higher = more creative |
| `top_p` | 0.92 | Nucleus sampling — only consider tokens covering 92% of probability mass |
| `top_k` | 50 | Limit to 50 most likely tokens at each step |
| `repetition_penalty` | 1.3 | Discourages the model from repeating the same phrases |
| `max_new_tokens` | 150 | Maximum tokens to generate per response |